
# Anterior-bulb maximum-opening boundary pipeline

This notebook builds a **recording-specific consensus lumen boundary** from several maximum-contraction frames.

It follows the file conventions used by the previous trajectory notebook:

- H5 video: frames stored in an image dataset, usually the first dataset in the file.
- Centerline CSV: `frame_id`, `node_id`, `x_pixel`, and `y_pixel`.
- Pixel size defaults to `0.04 µm/pixel` and can be changed below.

## Workflow

1. Select the H5 video, centerline CSV, and output folder.
2. Select one continuous frame range or enter several frame ranges manually.
3. Click the center of the anterior bulb on one reference frame. This creates the shared longitudinal origin, `s = 0`.
4. The notebook straightens every selected frame into centerline coordinates.
5. Trace the two visible lumen boundaries on each straightened frame.
6. The notebook interpolates every trace, calculates the pointwise median and interquartile range, plots the result, and exports CSV/NPZ/PNG files.

## OpenCV tracing controls

- **Left click:** add a point to the active boundary.
- **Right click:** undo the last point on the active boundary.
- **T:** trace the visually top boundary.
- **B:** trace the visually bottom boundary.
- **C:** clear the active boundary.
- **R:** clear both boundaries.
- **S or Enter:** accept the current frame.
- **K:** skip the current frame.
- **Esc/Q:** stop tracing and retain frames already accepted.

The pipeline calculates a two-dimensional visible-lumen template. The later normalized transverse coordinate should therefore be reported as **normalized in-plane transverse position**, `ρ₂D`, rather than a true three-dimensional radial coordinate.


In [ ]:

# Optional installation, only if a package is missing:
# %pip install h5py opencv-python numpy pandas scipy matplotlib ipywidgets

from __future__ import annotations

import json
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
try:
    from ipywidgets import (
        BoundedIntText,
        Button,
        HBox,
        IntRangeSlider,
        IntSlider,
        Layout,
        Output,
        Text,
        VBox,
    )
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
from scipy.interpolate import PchipInterpolator
from scipy.ndimage import gaussian_filter1d

import tkinter as tk
from tkinter import filedialog, messagebox

# ---------------------------------------------------------------------
# USER CONFIGURATION
# ---------------------------------------------------------------------
UM_PER_PIXEL = 0.04

# Straightened longitudinal window relative to the clicked bulb center.
S_MIN_UM = -7.0
S_MAX_UM = 7.0
S_STEP_UM = 0.04

# Straightened transverse window.
Y_MIN_UM = -3.0
Y_MAX_UM = 3.0
Y_STEP_UM = 0.04

# Display normalization for each straightened frame.
DISPLAY_LOW_PERCENTILE = 2.0
DISPLAY_HIGH_PERCENTILE = 99.5
USE_CLAHE = True
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)
DISPLAY_SCALE = 3.0

# Smoothing applied after PCHIP interpolation of the manually clicked points.
BOUNDARY_SMOOTH_SIGMA_UM = 0.08

# Consensus output requires at least this many manual traces at a given s.
MIN_VALID_TRACES_PER_S = 3

# The default H5 behavior matches the previous notebook: use the first root key.
# Set a full dataset path such as "data/video" to override the automatic choice.
H5_DATASET_KEY_OVERRIDE = None

print("Configuration loaded.")



## Helper functions

These functions load the existing file format, create a subpixel centerline coordinate system, straighten the lumen, and support interactive annotation.


In [ ]:

def make_hidden_tk_root() -> tk.Tk:
    root = tk.Tk()
    root.attributes("-topmost", True)
    root.withdraw()
    return root


def find_h5_image_datasets(h5_file: h5py.File) -> List[str]:
    """Return datasets that can plausibly contain a frame sequence."""
    candidates: List[str] = []

    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset) and obj.ndim >= 3:
            candidates.append(name)

    h5_file.visititems(visitor)
    return candidates


def choose_h5_dataset(h5_file: h5py.File, override: Optional[str] = None) -> str:
    if override is not None:
        if override not in h5_file:
            raise KeyError(f"H5_DATASET_KEY_OVERRIDE={override!r} was not found.")
        return override

    # First try the same convention as the previous trajectory notebook.
    root_keys = list(h5_file.keys())
    if root_keys:
        first = root_keys[0]
        if isinstance(h5_file[first], h5py.Dataset) and h5_file[first].ndim >= 3:
            return first

    candidates = find_h5_image_datasets(h5_file)
    if not candidates:
        raise ValueError("No H5 dataset with at least 3 dimensions was found.")
    return candidates[0]


def read_h5_frame(dataset: h5py.Dataset, frame_id: int) -> np.ndarray:
    """Read one frame and squeeze singleton channel dimensions."""
    frame = np.asarray(dataset[frame_id])
    frame = np.squeeze(frame)
    if frame.ndim == 3:
        # Keep a normal image channel arrangement if present; otherwise take first plane.
        if frame.shape[-1] in (3, 4):
            frame = cv2.cvtColor(frame[..., :3], cv2.COLOR_RGB2GRAY)
        elif frame.shape[0] in (3, 4):
            frame = cv2.cvtColor(np.moveaxis(frame[:3], 0, -1), cv2.COLOR_RGB2GRAY)
        else:
            frame = frame[0]
    if frame.ndim != 2:
        raise ValueError(f"Frame {frame_id} could not be converted to 2D; shape={frame.shape}")
    return frame.astype(np.float32, copy=False)


def robust_uint8(image: np.ndarray,
                 low_percentile: float = DISPLAY_LOW_PERCENTILE,
                 high_percentile: float = DISPLAY_HIGH_PERCENTILE,
                 use_clahe: bool = USE_CLAHE) -> np.ndarray:
    """Robustly normalize a grayscale image for manual display."""
    arr = np.asarray(image, dtype=np.float32)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros(arr.shape, dtype=np.uint8)
    lo, hi = np.nanpercentile(arr[finite], [low_percentile, high_percentile])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.nanmin(arr[finite]))
        hi = float(np.nanmax(arr[finite]))
    if hi <= lo:
        out = np.zeros(arr.shape, dtype=np.uint8)
    else:
        out = np.clip((arr - lo) / (hi - lo), 0, 1)
        out = np.nan_to_num(out, nan=0.0)
        out = np.round(out * 255).astype(np.uint8)
    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID)
        out = clahe.apply(out)
    return out


def load_centerline_csv(path: str | Path) -> Tuple[pd.DataFrame, Dict[int, pd.DataFrame]]:
    df = pd.read_csv(path)
    required = {"frame_id", "node_id", "x_pixel", "y_pixel"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Centerline CSV is missing columns: {sorted(missing)}")
    df = df.copy()
    df["frame_id"] = df["frame_id"].astype(int)
    lookup = {
        int(frame_id): group.sort_values("node_id").reset_index(drop=True)
        for frame_id, group in df.groupby("frame_id", sort=True)
    }
    return df, lookup


def project_point_to_centerline_nodes(centerline_df: pd.DataFrame,
                                      point_xy: Sequence[float]) -> Tuple[np.ndarray, float, float]:
    """
    Project a point onto a piecewise-linear centerline.

    Returns
    -------
    projected_xy : ndarray shape (2,)
    fractional_node_value : interpolated node_id coordinate
    cumulative_arc_px : arc length from the first centerline node to the projection
    """
    pts = centerline_df[["x_pixel", "y_pixel"]].to_numpy(dtype=float)
    node_values = centerline_df["node_id"].to_numpy(dtype=float)
    target = np.asarray(point_xy, dtype=float)

    if len(pts) < 2:
        raise ValueError("A centerline needs at least two points.")

    seg = pts[1:] - pts[:-1]
    seg_len_sq = np.sum(seg * seg, axis=1)
    safe = seg_len_sq > 0
    t = np.zeros(len(seg), dtype=float)
    t[safe] = np.sum((target - pts[:-1][safe]) * seg[safe], axis=1) / seg_len_sq[safe]
    t = np.clip(t, 0.0, 1.0)
    projections = pts[:-1] + t[:, None] * seg
    dist_sq = np.sum((projections - target) ** 2, axis=1)
    idx = int(np.argmin(dist_sq))

    projected = projections[idx]
    node_value = node_values[idx] + t[idx] * (node_values[idx + 1] - node_values[idx])

    seg_len = np.sqrt(seg_len_sq)
    cumulative = np.concatenate([[0.0], np.cumsum(seg_len)])
    arc_px = cumulative[idx] + t[idx] * seg_len[idx]
    return projected, float(node_value), float(arc_px)


def anchor_arc_from_node_value(centerline_df: pd.DataFrame, anchor_node_value: float) -> float:
    """Map the shared fractional node coordinate to arc length in one frame."""
    node_values = centerline_df["node_id"].to_numpy(dtype=float)
    pts = centerline_df[["x_pixel", "y_pixel"]].to_numpy(dtype=float)
    if anchor_node_value < node_values.min() or anchor_node_value > node_values.max():
        raise ValueError(
            f"Anchor node value {anchor_node_value:.3f} is outside frame node range "
            f"[{node_values.min():.3f}, {node_values.max():.3f}]."
        )
    seg_len = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    cumulative = np.concatenate([[0.0], np.cumsum(seg_len)])
    return float(np.interp(anchor_node_value, node_values, cumulative))


def build_straightening_map(centerline_df: pd.DataFrame,
                            anchor_node_value: float,
                            s_grid_um: np.ndarray,
                            y_grid_um: np.ndarray) -> Tuple[np.ndarray, np.ndarray, dict]:
    """Build image-coordinate remap arrays for one straightened frame."""
    pts = centerline_df[["x_pixel", "y_pixel"]].to_numpy(dtype=float)
    if len(pts) < 4:
        raise ValueError("Too few centerline points for straightening.")

    seg_len = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    keep = np.concatenate([[True], seg_len > 1e-9])
    pts = pts[keep]
    seg_len = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    cumulative_px = np.concatenate([[0.0], np.cumsum(seg_len)])

    # Shape-preserving interpolation avoids spline overshoot near the bulb.
    x_of_arc = PchipInterpolator(cumulative_px, pts[:, 0], extrapolate=False)
    y_of_arc = PchipInterpolator(cumulative_px, pts[:, 1], extrapolate=False)

    anchor_arc_px = anchor_arc_from_node_value(centerline_df, anchor_node_value)
    target_arc_px = anchor_arc_px + s_grid_um / UM_PER_PIXEL

    center_x = x_of_arc(target_arc_px)
    center_y = y_of_arc(target_arc_px)
    dx = x_of_arc.derivative()(target_arc_px)
    dy = y_of_arc.derivative()(target_arc_px)
    tangent_norm = np.hypot(dx, dy)
    tangent_norm[tangent_norm == 0] = np.nan
    tx = dx / tangent_norm
    ty = dy / tangent_norm

    # Left-handed image coordinates are acceptable because later analysis uses |rho|.
    nx = -ty
    ny = tx

    y_px = y_grid_um / UM_PER_PIXEL
    map_x = center_x[None, :] + y_px[:, None] * nx[None, :]
    map_y = center_y[None, :] + y_px[:, None] * ny[None, :]

    metadata = {
        "center_x": center_x,
        "center_y": center_y,
        "normal_x": nx,
        "normal_y": ny,
        "target_arc_px": target_arc_px,
        "valid_s": np.isfinite(center_x) & np.isfinite(center_y),
    }
    return map_x.astype(np.float32), map_y.astype(np.float32), metadata


def straighten_frame(frame: np.ndarray,
                     centerline_df: pd.DataFrame,
                     anchor_node_value: float,
                     s_grid_um: np.ndarray,
                     y_grid_um: np.ndarray) -> Tuple[np.ndarray, dict]:
    map_x, map_y, metadata = build_straightening_map(
        centerline_df, anchor_node_value, s_grid_um, y_grid_um
    )
    invalid_map = ~np.isfinite(map_x) | ~np.isfinite(map_y)
    safe_map_x = np.where(invalid_map, -1.0e6, map_x).astype(np.float32)
    safe_map_y = np.where(invalid_map, -1.0e6, map_y).astype(np.float32)
    patch = cv2.remap(
        frame.astype(np.float32),
        safe_map_x,
        safe_map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=float("nan"),
    )
    patch[invalid_map] = np.nan
    return patch, metadata


def parse_frame_spec(spec: str, available_frames: Sequence[int]) -> List[int]:
    """Parse strings such as '750-760, 812, 900-910:2'."""
    available = set(int(v) for v in available_frames)
    frames: List[int] = []
    spec = spec.strip()
    if not spec:
        return frames

    for token in spec.split(","):
        token = token.strip()
        if not token:
            continue
        step = 1
        if ":" in token:
            token, step_text = token.split(":", 1)
            step = max(1, int(step_text.strip()))
        match = re.fullmatch(r"(-?\d+)\s*-\s*(-?\d+)", token)
        if match:
            a, b = map(int, match.groups())
            direction = 1 if b >= a else -1
            local_step = step * direction
            frames.extend(range(a, b + direction, local_step))
        else:
            frames.append(int(token))

    return sorted({f for f in frames if f in available})


def click_centerline_anchor(frame: np.ndarray,
                            centerline_df: pd.DataFrame,
                            frame_id: int) -> Tuple[np.ndarray, float]:
    """Open an OpenCV window and let the user click the anterior-bulb center."""
    base = robust_uint8(frame, use_clahe=False)
    display_img = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
    pts = centerline_df[["x_pixel", "y_pixel"]].to_numpy(dtype=np.int32)
    cv2.polylines(display_img, [pts.reshape(-1, 1, 2)], False, (255, 255, 0), 1)

    state = {"click": None, "projected": None, "node_value": None}
    window = f"Select anterior-bulb center | frame {frame_id}"

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            projected, node_value, _ = project_point_to_centerline_nodes(centerline_df, (x, y))
            state["click"] = np.array([x, y], dtype=float)
            state["projected"] = projected
            state["node_value"] = node_value

    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(window, on_mouse)

    while True:
        canvas = display_img.copy()
        cv2.putText(
            canvas,
            "Click bulb center; ENTER/S accept; R reset; ESC cancel",
            (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2, cv2.LINE_AA,
        )
        if state["click"] is not None:
            p = tuple(np.round(state["click"]).astype(int))
            q = tuple(np.round(state["projected"]).astype(int))
            cv2.circle(canvas, p, 5, (0, 255, 0), -1)
            cv2.circle(canvas, q, 6, (0, 0, 255), 2)
            cv2.line(canvas, p, q, (0, 255, 255), 1)
        cv2.imshow(window, canvas)
        key = cv2.waitKey(20) & 0xFF
        if key in (13, 10, ord("s")) and state["projected"] is not None:
            break
        if key == ord("r"):
            state = {"click": None, "projected": None, "node_value": None}
        if key in (27, ord("q")):
            cv2.destroyWindow(window)
            cv2.waitKey(1)
            raise RuntimeError("Anchor selection cancelled.")

    cv2.destroyWindow(window)
    cv2.waitKey(1)
    return np.asarray(state["projected"], dtype=float), float(state["node_value"])


class BoundaryAnnotator:
    """OpenCV point-click annotator for the two visible boundaries."""

    def __init__(self, image_uint8: np.ndarray, frame_id: int, display_scale: float = 3.0):
        self.image = image_uint8
        self.frame_id = int(frame_id)
        self.scale = float(max(1.0, display_scale))
        self.mode = "top"
        self.points = {"top": [], "bottom": []}
        self.status = None
        self.window = f"Lumen boundary tracing | frame {self.frame_id}"

    def _scaled_shape(self) -> Tuple[int, int]:
        h, w = self.image.shape[:2]
        # Fit comfortably on an ordinary display.
        scale = min(self.scale, 1500 / max(w, 1), 850 / max(h, 1))
        self.actual_scale = max(scale, 1.0)
        return int(round(w * self.actual_scale)), int(round(h * self.actual_scale))

    def _to_original(self, x: int, y: int) -> Tuple[float, float]:
        return x / self.actual_scale, y / self.actual_scale

    def _to_display_points(self, pts: Sequence[Sequence[float]]) -> np.ndarray:
        arr = np.asarray(pts, dtype=float)
        return np.round(arr * self.actual_scale).astype(np.int32)

    def _mouse(self, event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            self.points[self.mode].append(self._to_original(x, y))
        elif event == cv2.EVENT_RBUTTONDOWN and self.points[self.mode]:
            self.points[self.mode].pop()

    def _draw_boundary(self, canvas: np.ndarray, name: str, color: Tuple[int, int, int]):
        pts = self.points[name]
        if not pts:
            return
        disp = self._to_display_points(pts)
        for p in disp:
            cv2.circle(canvas, tuple(p), 4, color, -1)
        if len(disp) >= 2:
            order = np.argsort(disp[:, 0])
            cv2.polylines(canvas, [disp[order].reshape(-1, 1, 2)], False, color, 2)

    def run(self) -> Optional[dict]:
        w, h = self._scaled_shape()
        cv2.namedWindow(self.window, cv2.WINDOW_NORMAL)
        cv2.setMouseCallback(self.window, self._mouse)

        while True:
            base = cv2.resize(self.image, (w, h), interpolation=cv2.INTER_NEAREST)
            canvas = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)

            # Display centerline row y=0, which may not be the geometric lumen midpoint.
            center_row = int(round((-Y_MIN_UM) / (Y_MAX_UM - Y_MIN_UM) * (self.image.shape[0] - 1)))
            center_row = int(round(center_row * self.actual_scale))
            cv2.line(canvas, (0, center_row), (w - 1, center_row), (255, 255, 0), 1)

            self._draw_boundary(canvas, "top", (0, 255, 255))
            self._draw_boundary(canvas, "bottom", (255, 0, 255))

            cv2.rectangle(canvas, (0, 0), (w, 72), (20, 20, 20), -1)
            cv2.putText(
                canvas,
                f"Frame {self.frame_id} | active: {self.mode.upper()} | "
                f"top={len(self.points['top'])}, bottom={len(self.points['bottom'])}",
                (12, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1, cv2.LINE_AA,
            )
            cv2.putText(
                canvas,
                "Left:add  Right:undo  T/B:mode  C:clear  R:reset  S/Enter:save  K:skip  Esc:stop",
                (12, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0, 255, 255), 1, cv2.LINE_AA,
            )
            cv2.imshow(self.window, canvas)
            key = cv2.waitKey(20) & 0xFF

            if key == ord("t"):
                self.mode = "top"
            elif key == ord("b"):
                self.mode = "bottom"
            elif key == ord("c"):
                self.points[self.mode] = []
            elif key == ord("r"):
                self.points = {"top": [], "bottom": []}
            elif key in (13, 10, ord("s")):
                if len(self.points["top"]) >= 3 and len(self.points["bottom"]) >= 3:
                    self.status = "saved"
                    break
                print(f"Frame {self.frame_id}: add at least 3 points to each boundary before saving.")
            elif key == ord("k"):
                self.status = "skipped"
                break
            elif key in (27, ord("q")):
                self.status = "stopped"
                break

        cv2.destroyWindow(self.window)
        cv2.waitKey(1)
        if self.status == "saved":
            return {
                "frame_id": self.frame_id,
                "top_points_col_row": self.points["top"],
                "bottom_points_col_row": self.points["bottom"],
            }
        return None


def pixel_points_to_physical(points_col_row: Sequence[Sequence[float]],
                             s_grid_um: np.ndarray,
                             y_grid_um: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(points_col_row, dtype=float)
    if arr.ndim != 2 or arr.shape[1] != 2:
        raise ValueError("Boundary points must have shape (N, 2).")
    col = arr[:, 0]
    row = arr[:, 1]
    s = np.interp(col, np.arange(len(s_grid_um)), s_grid_um)
    y = np.interp(row, np.arange(len(y_grid_um)), y_grid_um)
    return s, y


def fit_manual_curve(points_col_row: Sequence[Sequence[float]],
                     s_grid_um: np.ndarray,
                     y_grid_um: np.ndarray,
                     smooth_sigma_um: float = BOUNDARY_SMOOTH_SIGMA_UM) -> np.ndarray:
    s_pts, y_pts = pixel_points_to_physical(points_col_row, s_grid_um, y_grid_um)
    order = np.argsort(s_pts)
    s_pts, y_pts = s_pts[order], y_pts[order]

    # Average repeated x coordinates created by nearly vertical clicking.
    rounded = np.round(s_pts, 6)
    unique_s = np.unique(rounded)
    averaged_y = np.array([y_pts[rounded == val].mean() for val in unique_s])
    if len(unique_s) < 3:
        raise ValueError("At least three unique longitudinal positions are required.")

    curve = np.full_like(s_grid_um, np.nan, dtype=float)
    support = (s_grid_um >= unique_s.min()) & (s_grid_um <= unique_s.max())
    interpolator = PchipInterpolator(unique_s, averaged_y, extrapolate=False)
    curve[support] = interpolator(s_grid_um[support])

    if smooth_sigma_um > 0 and support.sum() >= 5:
        sigma_points = smooth_sigma_um / np.median(np.diff(s_grid_um))
        curve[support] = gaussian_filter1d(curve[support], sigma=max(sigma_points, 0.01), mode="nearest")
    return curve


def save_json_atomic(data: dict, path: Path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2), encoding="utf-8")
    tmp.replace(path)


print("Helper functions ready.")


## Select input files and load the recording

In [ ]:

root = make_hidden_tk_root()

input_h5_path = filedialog.askopenfilename(
    title="Select original H5 video file",
    filetypes=[("H5 files", "*.h5"), ("All files", "*.*")],
)
centerline_csv_path = filedialog.askopenfilename(
    title="Select centerline CSV (pharynx_axis_coordinates.csv)",
    filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
)
output_folder = filedialog.askdirectory(title="Select output folder")
root.destroy()

if not input_h5_path or not centerline_csv_path or not output_folder:
    raise RuntimeError("File selection cancelled or incomplete.")

output_dir = Path(output_folder)
output_dir.mkdir(parents=True, exist_ok=True)

centerline_df, centerline_lookup = load_centerline_csv(centerline_csv_path)

with h5py.File(input_h5_path, "r") as h5:
    h5_dataset_key = choose_h5_dataset(h5, H5_DATASET_KEY_OVERRIDE)
    dataset_shape = tuple(h5[h5_dataset_key].shape)
    n_video_frames = int(dataset_shape[0])

available_frames = sorted(
    f for f in centerline_lookup.keys()
    if 0 <= int(f) < n_video_frames
)

if not available_frames:
    raise RuntimeError("No centerline frames overlap the H5 frame range.")

print(f"H5: {input_h5_path}")
print(f"Dataset: {h5_dataset_key}  shape={dataset_shape}")
print(f"Centerline CSV: {centerline_csv_path}")
print(f"Overlapping frames: {len(available_frames)}  ({available_frames[0]} to {available_frames[-1]})")
print(f"Output folder: {output_dir}")



## Select maximum-contraction frames

Use the range and step controls for one continuous interval. For disjoint ranges, enter an optional frame specification such as:

`750-760, 812, 900-910:2`

The `:2` means every second frame. Only frames that exist in both the H5 and centerline CSV are retained.


In [ ]:
selected_frames: List[int] = []

if WIDGETS_AVAILABLE:
    frame_range_widget = IntRangeSlider(
        value=(available_frames[0], min(available_frames[0] + 10, available_frames[-1])),
        min=available_frames[0],
        max=available_frames[-1],
        step=1,
        description="Frame range",
        continuous_update=False,
        layout=Layout(width="850px"),
    )
    frame_step_widget = BoundedIntText(value=1, min=1, max=1000, description="Step")
    frame_spec_widget = Text(
        value="",
        placeholder="Optional: 750-760, 812, 900-910:2",
        description="Frame spec",
        layout=Layout(width="700px"),
    )
    preview_frame_widget = IntSlider(
        value=frame_range_widget.value[0],
        min=available_frames[0],
        max=available_frames[-1],
        step=1,
        description="Preview",
        continuous_update=False,
        layout=Layout(width="850px"),
    )
    confirm_frames_button = Button(description="Confirm selected frames", button_style="success")
    preview_output = Output()
    selection_output = Output()

    def nearest_available_frame(requested: int) -> int:
        arr = np.asarray(available_frames)
        return int(arr[np.argmin(np.abs(arr - requested))])

    def show_preview(frame_id: int):
        frame_id = nearest_available_frame(frame_id)
        with h5py.File(input_h5_path, "r") as h5:
            frame = read_h5_frame(h5[h5_dataset_key], frame_id)
        pts = centerline_lookup[frame_id][["x_pixel", "y_pixel"]].to_numpy()
        with preview_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.imshow(robust_uint8(frame, use_clahe=False), cmap="gray")
            ax.plot(pts[:, 0], pts[:, 1], linewidth=1)
            ax.set_title(f"Frame {frame_id}")
            ax.set_axis_off()
            plt.show()

    def on_preview_change(change):
        if change.get("name") == "value":
            show_preview(change["new"])

    def on_range_change(change):
        if change.get("name") == "value":
            preview_frame_widget.value = nearest_available_frame(change["new"][0])

    def confirm_frames(_):
        global selected_frames
        if frame_spec_widget.value.strip():
            frames = parse_frame_spec(frame_spec_widget.value, available_frames)
        else:
            start, end = frame_range_widget.value
            step = int(frame_step_widget.value)
            frames = [f for f in available_frames if start <= f <= end][::step]
        selected_frames = sorted(set(frames))
        with selection_output:
            clear_output(wait=True)
            if not selected_frames:
                print("No valid frames selected.")
            else:
                print(f"Selected {len(selected_frames)} frames:")
                print(selected_frames)

    preview_frame_widget.observe(on_preview_change, names="value")
    frame_range_widget.observe(on_range_change, names="value")
    confirm_frames_button.on_click(confirm_frames)

    display(VBox([
        frame_range_widget,
        HBox([frame_step_widget]),
        frame_spec_widget,
        preview_frame_widget,
        confirm_frames_button,
        selection_output,
        preview_output,
    ]))
    show_preview(preview_frame_widget.value)
else:
    print("ipywidgets is not installed, so the Tkinter fallback selector will be used.")
    root = make_hidden_tk_root()
    from tkinter import simpledialog
    spec = simpledialog.askstring(
        "Maximum-contraction frames",
        "Enter frames, e.g. 750-760, 812, 900-910:2",
        initialvalue=f"{available_frames[0]}-{min(available_frames[0] + 10, available_frames[-1])}",
        parent=root,
    )
    root.destroy()
    if spec is None:
        raise RuntimeError("Frame selection cancelled.")
    selected_frames = parse_frame_spec(spec, available_frames)
    if not selected_frames:
        raise RuntimeError("The frame specification produced no valid frames.")
    print(f"Selected {len(selected_frames)} frames:")
    print(selected_frames)



## Select the shared longitudinal origin

Run the following cell **after pressing “Confirm selected frames.”** Click the center of the anterior bulb on the reference frame. The clicked point is projected to the centerline, and its fractional `node_id` coordinate is transferred to all selected frames.


In [ ]:

if not selected_frames:
    raise RuntimeError("Confirm a non-empty frame selection in the previous cell first.")

reference_frame_id = selected_frames[len(selected_frames) // 2]
with h5py.File(input_h5_path, "r") as h5:
    reference_frame = read_h5_frame(h5[h5_dataset_key], reference_frame_id)

anchor_xy_reference, anchor_node_value = click_centerline_anchor(
    reference_frame,
    centerline_lookup[reference_frame_id],
    reference_frame_id,
)

anchor_metadata = {
    "reference_frame_id": int(reference_frame_id),
    "anchor_x_pixel": float(anchor_xy_reference[0]),
    "anchor_y_pixel": float(anchor_xy_reference[1]),
    "anchor_fractional_node_id": float(anchor_node_value),
}
(output_dir / "anchor_metadata.json").write_text(json.dumps(anchor_metadata, indent=2), encoding="utf-8")

print(f"Reference frame: {reference_frame_id}")
print(f"Projected anchor: x={anchor_xy_reference[0]:.3f}, y={anchor_xy_reference[1]:.3f}")
print(f"Shared fractional node_id: {anchor_node_value:.6f}")


## Straighten the selected frames

In [ ]:

s_grid_um = np.arange(S_MIN_UM, S_MAX_UM + 0.5 * S_STEP_UM, S_STEP_UM)
y_grid_um = np.arange(Y_MIN_UM, Y_MAX_UM + 0.5 * Y_STEP_UM, Y_STEP_UM)

straightened_patches: Dict[int, np.ndarray] = {}
straightening_metadata: Dict[int, dict] = {}
failed_frames = {}

with h5py.File(input_h5_path, "r") as h5:
    dataset = h5[h5_dataset_key]
    for frame_id in selected_frames:
        try:
            frame = read_h5_frame(dataset, frame_id)
            patch, metadata = straighten_frame(
                frame,
                centerline_lookup[frame_id],
                anchor_node_value,
                s_grid_um,
                y_grid_um,
            )
            straightened_patches[frame_id] = patch
            straightening_metadata[frame_id] = metadata
        except Exception as exc:
            failed_frames[frame_id] = repr(exc)

print(f"Successfully straightened {len(straightened_patches)} / {len(selected_frames)} selected frames.")
if failed_frames:
    print("Failed frames:")
    for frame_id, reason in failed_frames.items():
        print(f"  {frame_id}: {reason}")

# Preview up to 12 patches.
preview_ids = list(straightened_patches)[:12]
if preview_ids:
    ncols = min(4, len(preview_ids))
    nrows = math.ceil(len(preview_ids) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.5 * nrows), squeeze=False)
    for ax, frame_id in zip(axes.flat, preview_ids):
        ax.imshow(
            robust_uint8(straightened_patches[frame_id]),
            cmap="gray",
            origin="lower",
            extent=[S_MIN_UM, S_MAX_UM, Y_MIN_UM, Y_MAX_UM],
            aspect="auto",
        )
        ax.axhline(0, linewidth=0.8)
        ax.set_title(f"Frame {frame_id}")
        ax.set_xlabel("s (µm)")
        ax.set_ylabel("transverse y (µm)")
    for ax in axes.flat[len(preview_ids):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()



## Trace both visible boundaries on every selected frame

Annotations are saved incrementally to `manual_boundary_points.json`. If the notebook stops, rerun this cell and existing accepted frames will be skipped unless `RETRACE_EXISTING=True`.


In [ ]:

TRACE_JSON_PATH = output_dir / "manual_boundary_points.json"
RETRACE_EXISTING = False

if TRACE_JSON_PATH.exists():
    annotation_store = json.loads(TRACE_JSON_PATH.read_text(encoding="utf-8"))
else:
    annotation_store = {
        "metadata": {
            "input_h5_path": str(input_h5_path),
            "centerline_csv_path": str(centerline_csv_path),
            "h5_dataset_key": h5_dataset_key,
            "um_per_pixel": UM_PER_PIXEL,
            "s_min_um": S_MIN_UM,
            "s_max_um": S_MAX_UM,
            "s_step_um": S_STEP_UM,
            "y_min_um": Y_MIN_UM,
            "y_max_um": Y_MAX_UM,
            "y_step_um": Y_STEP_UM,
            "anchor_fractional_node_id": float(anchor_node_value),
        },
        "frames": {},
    }

for frame_id, patch in straightened_patches.items():
    key = str(frame_id)
    if key in annotation_store["frames"] and not RETRACE_EXISTING:
        print(f"Frame {frame_id}: existing trace retained.")
        continue

    display_patch = robust_uint8(patch)
    annotator = BoundaryAnnotator(display_patch, frame_id, DISPLAY_SCALE)
    result = annotator.run()
    if result is not None:
        annotation_store["frames"][key] = result
        save_json_atomic(annotation_store, TRACE_JSON_PATH)
        print(f"Frame {frame_id}: saved.")
    elif annotator.status == "stopped":
        print(f"Frame {frame_id}: tracing stopped by user. Previously accepted frames were retained.")
        break
    else:
        print(f"Frame {frame_id}: skipped.")
        continue

print(f"Annotation file: {TRACE_JSON_PATH}")
print(f"Accepted traces: {len(annotation_store['frames'])}")


## Calculate the individual and consensus boundaries

In [ ]:

if not TRACE_JSON_PATH.exists():
    raise RuntimeError("No manual-boundary annotation file was found.")

annotation_store = json.loads(TRACE_JSON_PATH.read_text(encoding="utf-8"))
frame_annotations = annotation_store.get("frames", {})
if len(frame_annotations) < 2:
    raise RuntimeError("At least two accepted frame traces are required; three or more are recommended.")

individual_rows = []
frame_curves = {}

for frame_key, annotation in frame_annotations.items():
    frame_id = int(frame_key)
    curve_top = fit_manual_curve(
        annotation["top_points_col_row"], s_grid_um, y_grid_um
    )
    curve_bottom = fit_manual_curve(
        annotation["bottom_points_col_row"], s_grid_um, y_grid_um
    )

    both_valid = np.isfinite(curve_top) & np.isfinite(curve_bottom)
    lower = np.where(both_valid, np.minimum(curve_top, curve_bottom), np.nan)
    upper = np.where(both_valid, np.maximum(curve_top, curve_bottom), np.nan)

    frame_curves[frame_id] = {"lower": lower, "upper": upper}
    for s, lo, hi in zip(s_grid_um, lower, upper):
        if np.isfinite(lo) and np.isfinite(hi):
            individual_rows.append({
                "frame_id": frame_id,
                "s_um": float(s),
                "lower_y_um": float(lo),
                "upper_y_um": float(hi),
                "midpoint_y_um": float((lo + hi) / 2),
                "half_width_um": float((hi - lo) / 2),
                "width_um": float(hi - lo),
            })

individual_df = pd.DataFrame(individual_rows)
individual_csv_path = output_dir / "individual_lumen_boundaries_long.csv"
individual_df.to_csv(individual_csv_path, index=False)

frame_ids_used = sorted(frame_curves)
lower_stack = np.vstack([frame_curves[f]["lower"] for f in frame_ids_used])
upper_stack = np.vstack([frame_curves[f]["upper"] for f in frame_ids_used])

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    lower_median = np.nanmedian(lower_stack, axis=0)
    upper_median = np.nanmedian(upper_stack, axis=0)
    lower_q25 = np.nanpercentile(lower_stack, 25, axis=0)
    lower_q75 = np.nanpercentile(lower_stack, 75, axis=0)
    upper_q25 = np.nanpercentile(upper_stack, 25, axis=0)
    upper_q75 = np.nanpercentile(upper_stack, 75, axis=0)

n_valid = np.sum(np.isfinite(lower_stack) & np.isfinite(upper_stack), axis=0)
valid_consensus = n_valid >= min(MIN_VALID_TRACES_PER_S, len(frame_ids_used))

for arr in [lower_median, upper_median, lower_q25, lower_q75, upper_q25, upper_q75]:
    arr[~valid_consensus] = np.nan

midpoint_median = (lower_median + upper_median) / 2
half_width_median = (upper_median - lower_median) / 2
width_median = upper_median - lower_median

consensus_df = pd.DataFrame({
    "s_um": s_grid_um,
    "s_pixel": s_grid_um / UM_PER_PIXEL,
    "lower_y_um": lower_median,
    "upper_y_um": upper_median,
    "midpoint_y_um": midpoint_median,
    "half_width_um": half_width_median,
    "width_um": width_median,
    "lower_y_pixel": lower_median / UM_PER_PIXEL,
    "upper_y_pixel": upper_median / UM_PER_PIXEL,
    "midpoint_y_pixel": midpoint_median / UM_PER_PIXEL,
    "half_width_pixel": half_width_median / UM_PER_PIXEL,
    "lower_q25_um": lower_q25,
    "lower_q75_um": lower_q75,
    "upper_q25_um": upper_q25,
    "upper_q75_um": upper_q75,
    "n_valid_traces": n_valid,
})

consensus_csv_path = output_dir / "consensus_lumen_boundary.csv"
consensus_df.to_csv(consensus_csv_path, index=False)

npz_path = output_dir / "consensus_lumen_boundary.npz"
np.savez_compressed(
    npz_path,
    s_um=s_grid_um,
    lower_y_um=lower_median,
    upper_y_um=upper_median,
    midpoint_y_um=midpoint_median,
    half_width_um=half_width_median,
    width_um=width_median,
    lower_q25_um=lower_q25,
    lower_q75_um=lower_q75,
    upper_q25_um=upper_q25,
    upper_q75_um=upper_q75,
    n_valid_traces=n_valid,
    frame_ids=np.asarray(frame_ids_used, dtype=int),
    anchor_fractional_node_id=float(anchor_node_value),
    um_per_pixel=float(UM_PER_PIXEL),
)

summary_metadata = {
    "input_h5_path": str(input_h5_path),
    "centerline_csv_path": str(centerline_csv_path),
    "h5_dataset_key": h5_dataset_key,
    "frame_ids_used": frame_ids_used,
    "number_of_traced_frames": len(frame_ids_used),
    "anchor_fractional_node_id": float(anchor_node_value),
    "um_per_pixel": float(UM_PER_PIXEL),
    "s_window_um": [float(S_MIN_UM), float(S_MAX_UM)],
    "y_window_um": [float(Y_MIN_UM), float(Y_MAX_UM)],
    "boundary_smooth_sigma_um": float(BOUNDARY_SMOOTH_SIGMA_UM),
    "minimum_valid_traces_per_s": int(MIN_VALID_TRACES_PER_S),
}
(output_dir / "consensus_lumen_boundary_metadata.json").write_text(
    json.dumps(summary_metadata, indent=2), encoding="utf-8"
)

print(f"Individual traces: {individual_csv_path}")
print(f"Consensus CSV:     {consensus_csv_path}")
print(f"Consensus NPZ:     {npz_path}")


## Plot and save the consensus boundary

In [ ]:

# Median of independently normalized images is used for visualization only.
image_stack = []
for frame_id in frame_ids_used:
    if frame_id in straightened_patches:
        image_stack.append(robust_uint8(straightened_patches[frame_id]).astype(float) / 255.0)

median_display_image = np.nanmedian(np.stack(image_stack), axis=0) if image_stack else None

fig, ax = plt.subplots(figsize=(12, 5))
if median_display_image is not None:
    ax.imshow(
        median_display_image,
        cmap="gray",
        origin="lower",
        extent=[S_MIN_UM, S_MAX_UM, Y_MIN_UM, Y_MAX_UM],
        aspect="auto",
    )

for frame_id in frame_ids_used:
    ax.plot(s_grid_um, frame_curves[frame_id]["lower"], linewidth=0.7, alpha=0.22)
    ax.plot(s_grid_um, frame_curves[frame_id]["upper"], linewidth=0.7, alpha=0.22)

ax.fill_between(s_grid_um, lower_q25, lower_q75, alpha=0.2, label="Lower-boundary IQR")
ax.fill_between(s_grid_um, upper_q25, upper_q75, alpha=0.2, label="Upper-boundary IQR")
ax.plot(s_grid_um, lower_median, linewidth=2.2, label="Consensus lower boundary")
ax.plot(s_grid_um, upper_median, linewidth=2.2, label="Consensus upper boundary")
ax.plot(s_grid_um, midpoint_median, linestyle="--", linewidth=1.2, label="Geometric midpoint")
ax.axhline(0, linestyle=":", linewidth=1, label="Tracked centerline")
ax.axvline(0, linestyle="--", linewidth=1, label="Clicked bulb center")
ax.set_xlabel("Position along centerline, s (µm)")
ax.set_ylabel("Transverse coordinate, y (µm)")
ax.set_title(f"Anterior-bulb maximum-opening consensus boundary ({len(frame_ids_used)} frames)")
ax.legend(loc="best", fontsize=8)
ax.set_xlim(S_MIN_UM, S_MAX_UM)
ax.set_ylim(Y_MIN_UM, Y_MAX_UM)
plt.tight_layout()

boundary_plot_path = output_dir / "consensus_lumen_boundary_overlay.png"
fig.savefig(boundary_plot_path, dpi=300, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(s_grid_um, width_median, linewidth=2)
ax2.axvline(0, linestyle="--", linewidth=1)
ax2.set_xlabel("Position along centerline, s (µm)")
ax2.set_ylabel("Consensus visible lumen width (µm)")
ax2.set_title("Maximum-opening lumen width profile")
ax2.grid(alpha=0.25)
plt.tight_layout()

width_plot_path = output_dir / "consensus_lumen_width_profile.png"
fig2.savefig(width_plot_path, dpi=300, bbox_inches="tight")
plt.show()

if median_display_image is not None:
    median_image_path = output_dir / "median_straightened_maximum_opening.png"
    cv2.imwrite(str(median_image_path), np.round(median_display_image * 255).astype(np.uint8))
    print(f"Median straightened image: {median_image_path}")

print(f"Boundary overlay plot:    {boundary_plot_path}")
print(f"Width profile plot:       {width_plot_path}")


## Use the exported boundary to calculate `ρ₂D`

The notebook exports both visible boundaries, so two related normalizations are available.

### Centerline-based, side-specific normalization

This matches the previous trajectory pipeline, where particle transverse displacement is measured from the tracked centerline:

\[
|\rho_{2D,centerline}|=
\begin{cases}
\dfrac{y_p}{y_{upper}(s_p)}, & y_p\ge 0,\\[6pt]
\dfrac{-y_p}{-y_{lower}(s_p)}, & y_p<0.
\end{cases}
\]

Here, `ρ = 0` is the tracked centerline and `|ρ| = 1` is the visible wall on the particle's side. This is the recommended default when integrating the boundary with the existing trajectory output.

### Geometric-midpoint normalization

If the tracked centerline is visibly offset from the middle of the lumen, use:

\[
\rho_{2D,midpoint}=\frac{y_p-y_{mid}(s_p)}{R(s_p)},
\qquad
R(s)=\frac{y_{upper}(s)-y_{lower}(s)}{2}.
\]

The helper function below supports both definitions. Use absolute values when the anatomical sign of the transverse side is not consistently registered.


In [ ]:
def boundary_at_s(s_um: float | np.ndarray,
                  boundary_df: pd.DataFrame = consensus_df) -> pd.DataFrame:
    """Interpolate consensus boundary quantities at one or more s positions."""
    query = np.atleast_1d(np.asarray(s_um, dtype=float))
    valid = boundary_df.dropna(subset=["lower_y_um", "upper_y_um"]).sort_values("s_um")
    if len(valid) < 2:
        raise ValueError("Consensus boundary has too few valid positions.")

    s_source = valid["s_um"].to_numpy()
    result = pd.DataFrame({"s_um": query})
    for column in ["lower_y_um", "upper_y_um", "midpoint_y_um", "half_width_um", "width_um"]:
        result[column] = np.interp(
            query,
            s_source,
            valid[column].to_numpy(),
            left=np.nan,
            right=np.nan,
        )
    return result


def calculate_rho_2d(s_um: float | np.ndarray,
                     particle_y_um: float | np.ndarray,
                     method: str = "centerline_side",
                     absolute: bool = True,
                     boundary_df: pd.DataFrame = consensus_df) -> np.ndarray:
    """
    Calculate normalized in-plane transverse position.

    Parameters
    ----------
    s_um:
        Particle position along the same centerline coordinate used for the template.
    particle_y_um:
        Signed transverse particle position from the tracked centerline.
    method:
        "centerline_side" (recommended for the previous trajectory pipeline) or
        "midpoint" (normalization around the geometric midpoint of the two walls).
    absolute:
        Return |rho| when True.
    """
    s_arr = np.atleast_1d(np.asarray(s_um, dtype=float))
    y_arr = np.atleast_1d(np.asarray(particle_y_um, dtype=float))
    s_arr, y_arr = np.broadcast_arrays(s_arr, y_arr)
    local = boundary_at_s(s_arr.ravel(), boundary_df)

    lower = local["lower_y_um"].to_numpy()
    upper = local["upper_y_um"].to_numpy()

    if method == "centerline_side":
        positive_radius = upper
        negative_radius = -lower
        denominator = np.where(y_arr.ravel() >= 0, positive_radius, negative_radius)
        rho = np.abs(y_arr.ravel()) / denominator
        if not absolute:
            rho = np.sign(y_arr.ravel()) * rho
    elif method == "midpoint":
        midpoint = local["midpoint_y_um"].to_numpy()
        half_width = local["half_width_um"].to_numpy()
        rho = (y_arr.ravel() - midpoint) / half_width
        if absolute:
            rho = np.abs(rho)
    else:
        raise ValueError("method must be 'centerline_side' or 'midpoint'.")

    scale = denominator if method == "centerline_side" else half_width
    invalid = ~np.isfinite(rho) | ~np.isfinite(scale) | (scale <= 0)
    rho[invalid] = np.nan
    return rho.reshape(s_arr.shape)


# Example only:
# rho = calculate_rho_2d(
#     s_um=[0.5, 1.2],
#     particle_y_um=[0.4, -0.2],
#     method="centerline_side",
#     absolute=True,
# )
# print(rho)

print("Functions boundary_at_s() and calculate_rho_2d() are ready.")



## Quality-control recommendations

Before using one consensus template for all particle events in this recording:

1. Inspect the individual boundary curves over the median image.
2. Check the IQR and `n_valid_traces` columns in the consensus CSV.
3. Confirm that the maximum-opening shape does not differ systematically between trapped and non-trapped events.
4. Manually compare consensus-derived `ρ₂D` against event-specific wall clicks for a small validation subset.
5. Treat values near or above `|ρ₂D| = 1` as potential segmentation, registration, or particle-centroid errors and inspect them manually.

The boundary template is recording-specific. Build a separate template for each animal or recording rather than pooling raw outlines across animals.
